# L3: Multi-agent Customer Support Automation (Modernized for CrewAI 1.x + Fireworks AI)

In this lesson, you will learn about the key elements that make multi-agent systems powerful:
- **Role Playing & Focus**
- **Tools & Web Scraping** (`ScrapeWebsiteTool`)
- **Cooperation & Delegation** (`allow_delegation=True`)
- **Guardrails & Structured Outputs**

### Modern Upgrades in this Notebook:
- Powered by **Fireworks AI** (`deepseek-v4-flash-0731`) with high token limits and zero cold starts.
- Uses CrewAI's native `LLM` class connecting to Fireworks' high-throughput inference endpoint.
- Explicitly activates `allow_delegation=True` on the QA agent (required in CrewAI 1.x).
- Modern `CrewOutput` formatting (`result.raw` and `result.token_usage`).

In [ ]:
import os
import warnings
from dotenv import load_dotenv
from crewai import Agent, Task, Crew, LLM

warnings.filterwarnings('ignore')

# 1. Load Fireworks API Key from .env
load_dotenv(override=True)
fireworks_api_key = os.getenv("FIREWORKS_API_KEY", "").strip().strip('"\'')

if not fireworks_api_key or fireworks_api_key.startswith("your-"):
    raise ValueError(
        "FIREWORKS_API_KEY is missing or still a placeholder!\n"
        "Please add your actual Fireworks key to .env: FIREWORKS_API_KEY=fw_..."
    )

# 2. Initialize the modern LLM using Fireworks AI
# High-throughput, generous rate limits, and fractions of a cent per request
llm = LLM(
    model="openai/accounts/fireworks/models/deepseek-v4-flash-0731",
    base_url="https://api.fireworks.ai/inference/v1",
    api_key=fireworks_api_key,
    temperature=0.5
)
print("✓ Successfully connected to Fireworks AI!")

## 1. Role Playing, Focus and Cooperation

- **Support Agent**: Handles customer inquiries directly (`allow_delegation=False`).
- **Quality Assurance Agent**: Reviews the draft and delegates back to the support agent if any detail is missing or inaccurate (`allow_delegation=True`).

In [ ]:
support_agent = Agent(
    role="Senior Support Representative",
    goal="Be the most friendly and helpful support representative in your team",
    backstory=(
        "You work at crewAI (https://crewai.com) and are now working on providing "
        "support to {customer}, a super important customer for your company. "
        "You need to make sure that you provide the best support! "
        "Make sure to provide full, complete answers and make no assumptions."
    ),
    llm=llm,
    allow_delegation=False,
    verbose=True
)

# In CrewAI 1.x, allow_delegation defaults to False.
# Setting allow_delegation=True enables the QA agent to send feedback back to the Support agent!
support_quality_assurance_agent = Agent(
    role="Support Quality Assurance Specialist",
    goal="Get recognition for providing the best support quality assurance in your team",
    backstory=(
        "You work at crewAI (https://crewai.com) and are now working with your team "
        "on a request from {customer} ensuring that the support representative is "
        "providing the best support possible.\n"
        "You need to make sure that the support representative is providing full, "
        "complete answers, and make no assumptions."
    ),
    llm=llm,
    allow_delegation=True,  # Explicitly enabled for multi-agent collaboration
    verbose=True
)

## 2. Tools: Web Scraping Tool

In this lesson, the Support Agent receives a tool to scrape the official CrewAI documentation page.

In [ ]:
from crewai_tools import ScrapeWebsiteTool

# Instantiate the website scraper tool pointed at the CrewAI docs
docs_scrape_tool = ScrapeWebsiteTool(
    website_url="https://docs.crewai.com/en/enterprise/guides/kickoff-crew"
)
print("✓ ScrapeWebsiteTool initialized.")

## 3. Creating Tasks

- Notice we assign `tools=[docs_scrape_tool]` at the **Task level** (`inquiry_resolution`).
- The `quality_assurance_review` task has no tools—its purpose is pure evaluation and delegation.

In [ ]:
inquiry_resolution = Task(
    description=(
        "{customer} just reached out with a super important ask:\n"
        "{inquiry}\n\n"
        "{person} from {customer} is the one that reached out. "
        "Make sure to use the documentation tool to provide the best, factually grounded support possible.\n"
        "You must strive to provide a complete and accurate response to the customer's inquiry."
    ),
    expected_output=(
        "A detailed, informative response to the customer's inquiry that addresses "
        "all aspects of their question.\n"
        "The response should include references to everything you used to find the answer, "
        "including external data or solutions. "
        "Ensure the answer is complete, leaving no questions unanswered, and maintain "
        "a helpful and friendly tone throughout."
    ),
    tools=[docs_scrape_tool],
    agent=support_agent,
)

quality_assurance_review = Task(
    description=(
        "Review the response drafted by the Senior Support Representative for {customer}'s inquiry.\n"
        "Ensure that the answer is comprehensive, accurate, and adheres to the high-quality standards "
        "expected for customer support.\n"
        "Verify that all parts of the customer's inquiry have been addressed thoroughly, "
        "with a helpful and friendly tone.\n"
        "If the draft lacks critical details or is incomplete, delegate back with constructive feedback.\n"
        "Check for references and sources used to find the information."
    ),
    expected_output=(
        "A final, detailed, and informative response ready to be sent to the customer.\n"
        "This response should fully address the customer's inquiry, incorporating all "
        "relevant feedback and improvements.\n"
        "Don't be too formal—maintain a professional, friendly, and approachable tone throughout."
    ),
    agent=support_quality_assurance_agent,
)

## 4. Assembling and Running the Crew

- `verbose=True`: Standard boolean logging in CrewAI 1.x.
- Fireworks AI easily handles the multi-agent token payload from scraping and delegation.

In [ ]:
crew = Crew(
    agents=[support_agent, support_quality_assurance_agent],
    tasks=[inquiry_resolution, quality_assurance_review],
    verbose=True
)

inputs = {
    "customer": "DeepLearningAI",
    "person": "Andrew Ng",
    "inquiry": "I need help with setting up a Crew and kicking it off, specifically "
               "how can I add memory to my crew? Can you provide guidance?"
}

result = crew.kickoff(inputs=inputs)

# Render result cleanly with Markdown
from IPython.display import Markdown
Markdown(result.raw)

In [ ]:
# View execution metrics & token usage breakdown
print("Execution Token Usage Summary:")
print(result.token_usage)